# SkillLens: fine-tuned BERT skill tagger (Sprint 3)

Runs on **Google Colab with a free T4 GPU**. Menu: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

What it does:
1. Downloads SkillSpan (train/dev/test).
2. Trains **two** token taggers: one for `Skill` spans, one for `Knowledge` spans (SkillSpan labels them in separate columns, and they can overlap).
3. Predicts on **dev**, turns the BIO tags back into spans, and scores them with the **same strict/partial span F1** as our dictionary baseline (strict F1 0.308).
4. Saves `results_<model>.json` for the report and the MLflow log.

Run time: about 15–25 minutes per model on a T4. Change `MODEL_NAME` in the next cell to run the second model.

In [ ]:
# ---- settings (change MODEL_NAME for the second run) ----
MODEL_NAME = "bert-base-cased"          # second run: "jjzha/jobspanbert-base-cased"
EPOCHS = 3
LR = 5e-5
BATCH = 16
MAX_LEN = 256
SEED = 42

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4 GPU!)")

In [ ]:
!pip -q install transformers

In [ ]:
# ---- 1. download SkillSpan ----
import json, urllib.request
BASE = "https://raw.githubusercontent.com/kris927b/SkillSpan/main/data/json/"

def load(split):
    raw = urllib.request.urlopen(BASE + f"{split}.json").read().decode("utf-8")
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

train, dev = load("train"), load("dev")
print(len(train), "train sentences,", len(dev), "dev sentences")
print(train[0]["tokens"][:12], train[0]["tags_skill"][:12])

In [ ]:
# ---- 2. span helpers: same logic as src/skilllens/spans.py and evaluate.py ----
from collections import defaultdict

def bio_to_spans(tags, label):
    spans, start = [], None
    for i, t in enumerate(tags):
        if t.startswith("B") or (t.startswith("I") and start is None):
            if start is not None:
                spans.append((start, i, label))
            start = i
        elif t.startswith("I"):
            continue
        else:
            if start is not None:
                spans.append((start, i, label))
                start = None
    if start is not None:
        spans.append((start, len(tags), label))
    return spans

LABELS = ("Skill", "Knowledge")

def _prf(tp, n_pred, n_gold):
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4), "tp": tp, "pred": n_pred, "gold": n_gold}

def _matches(pred, gold, mode):
    if mode == "strict":
        return len(set(pred) & set(gold))
    used, tp = set(), 0
    for p in pred:
        for g in gold:
            if g not in used and g[2] == p[2] and p[0] < g[1] and g[0] < p[1]:
                used.add(g); tp += 1; break
    return tp

def evaluate(predictions, golds):
    out = {}
    for mode in ("strict", "partial"):
        c = defaultdict(lambda: [0, 0, 0])
        for pred, gold in zip(predictions, golds):
            for lab in LABELS:
                p = [s for s in pred if s[2] == lab]; g = [s for s in gold if s[2] == lab]
                c[lab][0] += _matches(p, g, mode); c[lab][1] += len(p); c[lab][2] += len(g)
        res = {lab: _prf(*c[lab]) for lab in LABELS}
        res["overall"] = _prf(*[sum(c[l][i] for l in LABELS) for i in range(3)])
        out[mode] = res
    return out

# quick self-check
assert bio_to_spans(["O", "B", "I", "O", "I"], "Skill") == [(1, 3, "Skill"), (4, 5, "Skill")]
print("helpers OK")

In [ ]:
# ---- 3. tokenise: label only the FIRST sub-word of each word, -100 elsewhere ----
import random, numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TAG2ID = {"O": 0, "B": 1, "I": 2}
ID2TAG = {v: k for k, v in TAG2ID.items()}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def norm(tag):            # SkillSpan tags look like "B", "I", "O" (sometimes with a suffix)
    return tag[0] if tag and tag[0] in "BIO" else "O"

def encode(sentences, column):
    enc = tokenizer([s["tokens"] for s in sentences], is_split_into_words=True,
                    truncation=True, max_length=MAX_LEN)
    all_labels = []
    for i, s in enumerate(sentences):
        word_ids, prev, labels = enc.word_ids(i), None, []
        for w in word_ids:
            labels.append(-100 if w is None or w == prev else TAG2ID[norm(s[column][w])])
            prev = w
        all_labels.append(labels)
    enc["labels"] = all_labels
    return enc

def batches(enc, idx):
    for b in range(0, len(idx), BATCH):
        ids = idx[b:b + BATCH]
        feats = [{k: enc[k][i] for k in enc.keys()} for i in ids]
        L = max(len(f["input_ids"]) for f in feats)
        out = {}
        for k in feats[0]:
            pad = -100 if k == "labels" else (tokenizer.pad_token_id if k == "input_ids" else 0)
            out[k] = torch.tensor([f[k] + [pad] * (L - len(f[k])) for f in feats]).to(DEVICE)
        yield out
print("tokenizer ready:", MODEL_NAME)

In [ ]:
# ---- 4. train one tagger per column, predict dev spans ----
import time

def train_and_predict(column, label):
    tr, dv = encode(train, column), encode(dev, column)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID2TAG, label2id=TAG2ID).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    for epoch in range(EPOCHS):
        model.train(); t0 = time.time(); order = list(range(len(train))); random.shuffle(order); total = 0.0
        for batch in batches(tr, order):
            loss = model(**batch).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); total += loss.item()
        print(f"[{label}] epoch {epoch + 1}/{EPOCHS} loss {total:.1f} ({time.time() - t0:.0f}s)")
    # predict dev
    model.eval(); preds = []
    with torch.no_grad():
        for start in range(0, len(dev), BATCH):
            idx = list(range(start, min(start + BATCH, len(dev))))
            batch = next(batches(dv, idx)); batch.pop("labels")
            logits = model(**batch).logits.argmax(-1).cpu().tolist()
            for j, i in enumerate(idx):
                tags, prev = ["O"] * len(dev[i]["tokens"]), None
                for pos, w in enumerate(dv.word_ids(i)):
                    if w is not None and w != prev:
                        tags[w] = ID2TAG[logits[j][pos]]
                    prev = w
                preds.append(bio_to_spans(tags, label))
    del model; torch.cuda.empty_cache()
    return preds

skill_pred = train_and_predict("tags_skill", "Skill")
know_pred = train_and_predict("tags_knowledge", "Knowledge")

In [ ]:
# ---- 5. score on dev with the same metric as the baseline ----
predictions = [s + k for s, k in zip(skill_pred, know_pred)]
golds = [bio_to_spans(s["tags_skill"], "Skill") + bio_to_spans(s["tags_knowledge"], "Knowledge") for s in dev]
res = evaluate(predictions, golds)

BASELINE = {"strict": 0.308, "partial": 0.418}   # dictionary baseline, dev, overall F1
print(f"Model: {MODEL_NAME}\n")
print(f"{'':10}{'label':11}{'P':>7}{'R':>7}{'F1':>7}")
for mode in ("strict", "partial"):
    for lab in ("Skill", "Knowledge", "overall"):
        m = res[mode][lab]
        print(f"{mode:10}{lab:11}{m['precision']:7.3f}{m['recall']:7.3f}{m['f1']:7.3f}")
    print(f"{'':10}{'baseline':11}{'':14}{BASELINE[mode]:7.3f}\n")

In [ ]:
# ---- 6. save + download the results file (put it in reports/ in the repo) ----
out = {"model": MODEL_NAME, "split": "dev", "epochs": EPOCHS, "lr": LR, "batch": BATCH,
       "max_len": MAX_LEN, "seed": SEED, "results": res,
       "examples": [{"tokens": dev[i]["tokens"], "gold": golds[i], "pred": predictions[i]} for i in range(30)]}
fname = f"results_{MODEL_NAME.split('/')[-1]}_dev.json"
with open(fname, "w") as f:
    json.dump(out, f, indent=1)
from google.colab import files
files.download(fname)